# The MONKEY challenge: Machine-learning for Optimal detection of iNflammatory cells in KidnEY transplant biopsies (Colab)

End-to-end run: install, mount Drive, copy the packed `nephro` data to the
local SSD, train leave-one-centre-out with checkpoint/resume, detect, score
out-of-fold, and regenerate figures.

## 1. Install

Pin the whole torch trio to one matching CUDA build in a single line, in its
own cell. After this cell finishes, restart the runtime (Runtime -> Restart
session) before running anything else, so Colab drops the preinstalled
torch libraries.

In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  --index-url https://download.pytorch.org/whl/cu126

In [ ]:
# Clone the public repository and install the package (editable, no deps) plus
# the pinned runtime dependencies. matplotlib is only needed for the figures.
# (Adjust the URL if you forked the repo under a different owner/name.)
!git clone https://github.com/m-logviniuk/monkey_challenge.git /content/monkey_challenge
%cd /content/monkey_challenge
!pip install -e . --no-deps
!pip install numpy==1.26.4 scipy==1.13.0 h5py==3.11.0 tqdm==4.66.4 matplotlib==3.8.4

Restart the session (Runtime -> Restart session) and skip to the next cell. Do not re-run the install cells.

In [ ]:
# PyTorch check: catches errors early before running the GPU.
import torch
print(torch.__version__)
print("cuda:", torch.cuda.is_available())

## 2. Mount Drive, set paths, create output dirs

Checkpoints and results live on Drive so they survive a disconnect; the data
is copied to the local SSD for fast reads.

In [ ]:
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
DATA_DIR = Path('/content/data/monkey')          # local SSD, fast reads
CKPT_DIR = DRIVE / 'monkey_challenge' / 'checkpoints'
RESULTS_DIR = DRIVE / 'monkey_challenge' / 'results'
for d in (DATA_DIR, CKPT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

os.environ['MONKEY_DATA_DIR'] = str(DATA_DIR)
os.environ['MONKEY_CKPT_DIR'] = str(CKPT_DIR)
os.environ['MONKEY_RESULTS_DIR'] = str(RESULTS_DIR)

In [ ]:
# Copy the MONKEY per-case cases from Drive to the local SSD for fast reads.
# The data lives under MyDrive/nephro/monkey/ as <case>.h5 files, with
# monkey_manifest.csv alongside in MyDrive/nephro/. Raw WSIs are never uploaded.
import shutil

SRC = DRIVE / 'nephro' / 'monkey'
h5_files = sorted(SRC.glob('*.h5'))
if not h5_files:
    raise FileNotFoundError(f'no *.h5 found under {SRC}; check the nephro upload.')

for h5 in h5_files:
    shutil.copy2(h5, DATA_DIR / h5.name)
manifest = DRIVE / 'nephro' / 'monkey_manifest.csv'
if manifest.exists():
    shutil.copy2(manifest, DATA_DIR / manifest.name)

print(f'copied {len(h5_files)} cases from {SRC} to {DATA_DIR}')
print(sorted(p.name for p in DATA_DIR.glob('*.h5'))[:5], '...')

## 3. Smoke check (no data, no weights)

In [ ]:
!monkey smoke

## 4. Train leave-one-centre-out (checkpoint/resume)

Four folds, one per centre (A/B/C/D). Each fold checkpoints after every epoch
to `fold_{i}_{tag}.pt` on Drive, so a disconnect is resumed by simply
re-running this cell. `tag` encodes the head and augmentation setting.

In [ ]:
!monkey train --device cuda

### Ablations (train the KAN and no-aug variants)

Two ablations are reported: the parameter-matched **KAN** decoder head and the
**without stain-augmentation** run. Each is selected through an environment flag
and writes to its own checkpoint tag (`kan_aug`, `conv_noaug`), so runs never
overwrite the main `conv_aug` weights. Train them here so the out-of-fold step
below can score all three configurations. If these checkpoints already exist on
Drive, re-running this cell simply resumes and is a no-op.

In [ ]:
# KAN head ablation (parameter-matched to the conv head).
os.environ['MONKEY_HEAD'] = 'kan'
!monkey train --device cuda
os.environ['MONKEY_HEAD'] = 'conv'

# Stain-augmentation ablation.
os.environ['MONKEY_USE_STAIN_AUG'] = '0'
!monkey train --device cuda
os.environ['MONKEY_USE_STAIN_AUG'] = '1'

## 5. Detect one case

Writes predicted level-0 points `(x0, y0, score, class)` to a CSV in
`results/`.

In [ ]:
case = sorted(DATA_DIR.glob('*.h5'))[0]
!monkey detect --case "$case" --fold 1 --device cuda

## 6. Out-of-fold evaluation

Pooled leave-one-centre-out FROC and precision/recall with bootstrap CIs. This
is inference only (no training). Score all three configurations into separate
files:

- `metrics.json` - main model (conv head, stain-aug);
- `metrics_noaug.json` - stain-augmentation ablation (conv head, no aug);
- `metrics_kan.json` - parameter-matched KAN-head ablation.

The environment flag picks the matching checkpoint tag; the conv results are
never recomputed or overwritten by the KAN/no-aug runs.

In [ ]:
# Main model (conv head, stain-aug).
!monkey oof --device cuda --out "$MONKEY_RESULTS_DIR/metrics.json"

# Stain-augmentation ablation (conv head, no aug).
!MONKEY_USE_STAIN_AUG=0 monkey oof --device cuda --out "$MONKEY_RESULTS_DIR/metrics_noaug.json"

# KAN-head ablation (parameter-matched to the conv head).
!MONKEY_HEAD=kan monkey oof --device cuda --out "$MONKEY_RESULTS_DIR/metrics_kan.json"

## 7. Figures

Regenerates the EDA, FROC/precision-recall/calibration, detection,
domain-generalization, and KAN interpretability figures into `results/`.

Two points:

- The **with/without stain-augmentation** panel of `fig_domain_generalization.png`
  is only drawn when `MONKEY_METRICS_NOAUG` points at the no-aug metrics file
  produced above. Without it, that panel is left blank.
- The **KAN interpretability** figures need a KAN-head checkpoint (`kan_aug`);
  they are skipped with a message if none is found.

Passing `--data-dir` and `--ckpt-dir` lets the detection and KAN figures load
the cases and weights.

In [ ]:
# MONKEY_METRICS_NOAUG is required for the stain-augmentation ablation panel.
!MONKEY_METRICS_NOAUG="$MONKEY_RESULTS_DIR/metrics_noaug.json" \
  monkey figures --out "$MONKEY_RESULTS_DIR" \
  --data-dir "$MONKEY_DATA_DIR" --ckpt-dir "$MONKEY_CKPT_DIR"

## Outputs on Drive

- `monkey_challenge/checkpoints/fold_{i}_{tag}.pt` - per-fold weights
  (`conv_aug`, `conv_noaug`, `kan_aug`);
- `monkey_challenge/results/metrics.json` - pooled out-of-fold metrics (conv);
- `monkey_challenge/results/metrics_noaug.json` - no-stain-aug ablation;
- `monkey_challenge/results/metrics_kan.json` - KAN-head ablation;
- `monkey_challenge/results/*.png` - figures;
- `monkey_challenge/results/*_points.csv` - predicted points per case.